# Generation Parameters

**Module:** 05 — LLM Fundamentals

Temperature, top-k/top-p, length/stop controls, penalties, and practical defaults.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Predict how temperature changes diversity
- Use top-k / top-p safely
- Set max tokens, stops, and penalties with intent
- Choose defaults for FAQ vs creative tasks


## Temperature

**Definition.** Temperature scales logits before softmax: lower → peakier (deterministic); higher → flatter (diverse).

**Why it matters.** Primary knob for creativity vs stability.

**How it works.** logits' = logits / T; sample from softmax(logits').

**Intuition.** T→0 argues for the mode; high T explores the tail.

**Common pitfalls.**
- T>1 on factual FAQ
- Assuming T=0 fully eliminates hallucinations

**When to use.** Low for extraction/FAQ; moderate for chat; higher for brainstorming.


In [ ]:
# Demo 1 — temperature effect
import numpy as np
logits = np.array([2.0, 1.0, 0.5])
def probs(t):
    x = logits / t
    p = np.exp(x - x.max()); return p/p.sum()
for t in [0.2, 0.7, 1.3]:
    print(t, probs(t).round(3))


In [ ]:
# Demo 2 — argmax as T→0
print("argmax token index", int(np.argmax(logits)))


### Try it yourself — Temperature

1. Pick T for: SQL generation, marketing slogans, legal clause extract.


## Top-k / Top-p (Nucleus)

**Definition.** **Top-k** keeps k highest logits; **top-p** keeps the smallest set with cumulative prob ≥ p.

**Why it matters.** Truncation avoids sampling bizarre tail tokens without forcing full greediness.

**How it works.** Filter distribution then renormalize and sample (often with temperature).

**Intuition.** Only consider plausible next words, not the whole dictionary.

**Common pitfalls.**
- Combining extreme T with extreme p poorly tuned
- k too small for code

**When to use.** Default safety rails for most chat apps.


In [ ]:
# Demo 1 — top-k filter
import numpy as np
p = np.array([0.4, 0.3, 0.2, 0.05, 0.05])
k=2
idx = np.argsort(-p)[:k]
filt = np.zeros_like(p); filt[idx]=p[idx]; filt/=filt.sum()
print(filt)


In [ ]:
# Demo 2 — nucleus top-p
p = np.array([0.4, 0.3, 0.15, 0.1, 0.05])
order = np.argsort(-p)
cum = np.cumsum(p[order])
keep = order[: np.searchsorted(cum, 0.9) + 1]
print("keep idx", keep, "mass", p[keep].sum())


In [ ]:
# Demo 3 — API params
print({"temperature": 0.7, "top_p": 0.9, "top_k": 40})


### Try it yourself — Top-k / Top-p (Nucleus)

1. When would top-p=0.95 with T=0.2 be preferable to T=1.2 unconstrained?


## Max Tokens, Stop Sequences, Presence/Frequency Penalties

**Definition.** Controls that bound length, force termination, and discourage repetition.

**Why it matters.** Prevents runaway cost and loops; shapes style.

**How it works.** Set max_tokens; stop on delimiters; apply penalties to logged probabilities of seen tokens.

**Intuition.** Brakes, guardrails, and anti-echo.

**Common pitfalls.**
- Stops that truncate JSON mid-string
- Heavy penalties harming needed repetition in code

**When to use.** Structured outputs, creative writing, long agents.


In [ ]:
# Demo 1 — stop sequences
text = "ANSWER:\n42\nEND\nextra"
stop = "\nEND"
print(text.split(stop)[0])


In [ ]:
# Demo 2 — frequency penalty sketch
from collections import Counter
toks = ["the", "cat", "the", "cat", "the"]
freq = Counter(toks)
penalty = 0.5
adjusted = {t: 1.0 - penalty*(c-1) for t,c in freq.items()}
print(adjusted)


In [ ]:
# Demo 3 — max tokens budget
prompt_toks, ctx_limit, max_new = 3000, 8192, 1024
print("ok", prompt_toks + max_new <= ctx_limit)


### Try it yourself — Max Tokens, Stop Sequences, Presence/Frequency Penalties

1. Design stops for a JSON object generator.


## Practical Guidance

**Definition.** A default recipe book for common task classes.

**Why it matters.** Teams waste weeks random-searching parameters without task-based defaults.

**How it works.** Start from templates; evaluate; change one knob at a time.

**Intuition.** Recipes first, gourmet tuning later.

**Common pitfalls.**
- Tuning on three cherry-picked prompts
- Different params per environment with no docs

**When to use.** Ship defaults with every prompt template version.

| Task | Temp | Notes |
|------|------|-------|
| Extraction | 0–0.2 | Deterministic |
| RAG FAQ | 0.1–0.3 | Cite/refuse |
| Chat | 0.5–0.8 | Natural |
| Creative | 0.8–1.2 | Diverse |


In [ ]:
# Demo 1 — defaults table
recipes = {
  "faq_rag": {"temperature": 0.1, "top_p": 1.0, "max_tokens": 400},
  "brainstorm": {"temperature": 0.9, "top_p": 0.95, "max_tokens": 800},
  "json_extract": {"temperature": 0.0, "top_p": 1.0, "max_tokens": 300},
}
for k,v in recipes.items():
    print(k, v)


In [ ]:
# Demo 2 — one-knob experiment log
log = []
log.append({"temp": 0.2, "exact_match": 0.71})
log.append({"temp": 0.5, "exact_match": 0.64})
print(log)


### Try it yourself — Practical Guidance

1. Publish a defaults YAML for your team with 3 task types.


## Glossary

- **nucleus sampling**: Top-p truncation
- **TTFT**: See inference notebook


### Workshop drill — Generation Parameters (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Generation Parameters
headings = ['Temperature', 'Top-k / Top-p (Nucleus)', 'Max Tokens, Stop Sequences, Presence/Frequency Penalties', 'Practical Guidance']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Generation Parameters (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Generation Parameters
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Generation Parameters (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Generation Parameters
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


### Workshop drill — Generation Parameters (4)

Write a tiny unit-testable helper related to the lesson and assert two cases.


In [ ]:
# Workshop drill 4 — Generation Parameters
def ok(x):
    return x is not None
assert ok(1) and not ok(None)
print('ok')


### Workshop drill — Generation Parameters (5)

Sketch an API request/response JSON for a realistic call tied to this topic.


In [ ]:
# Workshop drill 5 — Generation Parameters
import json
print(json.dumps({'model':'...','input':'...','output':'...'}, indent=2))


## Summary & Key Takeaways

- Temperature reshapes the distribution sharpness
- Top-k/p truncate tails
- Stops/max tokens control cost and structure
- Use task recipes; tune with evals

### Practice

A/B two temperatures on 20 prompts; record win rate.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
